In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report




data= pd.read_csv('../data/customer_support_tickets.csv')



data=data[["Ticket Subject",
"Ticket Description",
"Ticket Type"]] # remove irrelevant columns


ticket_type_map = {
"Refund request": "refund",
"Technical issue": "technical",
"Cancellation request": "cancellation",
"Product inquiry": "product",
"Billing inquiry": "billing"
}


data.rename(columns={"Ticket Type":"label",}, inplace=True)
data["label"]= data["label"].map(ticket_type_map)   # more beautification 
data["text"] = (
    "SUBJECT: " + data["Ticket Subject"].fillna("") +
    " DESCRIPTION: " + data["Ticket Description"].fillna("")
)
data.drop(columns=["Ticket Subject", "Ticket Description"], inplace=True) # remove subject and description columns



# remove placeholders

data["text"] = data["text"].str.replace(
    r"\{.*?\}", "", regex=True
)

# removing the greeting and closing statements
boilerplate = [
"I'm having an issue with",
"Please assist",
"Thank you",
"Thanks"
]

for phrase in boilerplate:
    data["text"] = data["text"].str.replace(
        phrase, "", regex=False
    )
data["text"] = data["text"].str.replace("\n", " ", regex=False)





x= data["text"]
y= data["label"]

X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.9,
    stop_words='english',
    sublinear_tf=True
)

x_train_vec = vectorizer.fit_transform(X_train)
x_test_vec = vectorizer.transform(X_test)

model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    n_jobs=-1
)

model.fit(x_train_vec, y_train)

y_pred = model.predict(x_test_vec)


2598
4587    cancellation
2794    cancellation
1679          refund
6863       technical
6395         product
Name: label, dtype: str
6510    product
4921     refund
871     billing
6086    product
3305     refund
Name: label, dtype: str


/Users/anishrajpandey/Code Playground/personal_projects/ticket-triage/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


In [9]:
print("Train nonzeros:", x_train_vec.nnz)
print("Test nonzeros:", x_test_vec.nnz)


print("Average nonzeros per train sample:", x_train_vec.nnz / x_train_vec.shape[0])
print("Average nonzeros per test sample:", x_test_vec.nnz / x_test_vec.shape[0])



Train nonzeros: 218461
Test nonzeros: 54158
Average nonzeros per train sample: 32.24516605166052
Average nonzeros per test sample: 31.97048406139315
